In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

### Boston예제

In [2]:
boston = fetch_openml(name='boston', version=1, as_frame=True, parser='auto')
X = boston.data
y = boston.target

In [3]:
X.head(2)

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.9,4.98
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.9,9.14


In [4]:
y[:10]

0    24.0
1    21.6
2    34.7
3    33.4
4    36.2
5    28.7
6    22.9
7    27.1
8    16.5
9    18.9
Name: MEDV, dtype: float64

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
# 8:1:1
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.111, random_state=42)

In [7]:
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train.values).view(-1,1) # 2차원 구조로 변환(N, 1)

X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val.values).view(-1,1)

X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test.values).view(-1,1)

C:\Users\Win11Pro\AppData\Local\Temp\ipykernel_3760\2981704590.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  y_train = torch.FloatTensor(y_train.values).view(-1,1) # 2차원 구조로 변환(N, 1)


In [8]:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [9]:
print(f"Train set : {X_train.shape}, Val set: {X_val.shape}, Test set : {X_test.shape}")

Train set : torch.Size([404, 13]), Val set: torch.Size([51, 13]), Test set : torch.Size([51, 13])


In [10]:
model = nn.Sequential(
    nn.Linear(13,64),
    nn.ReLU(),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,1)
)

In [11]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [12]:
epochs = 100
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = criterion(val_pred, y_val)

    if (epoch + 1) % 10 == 0:
        print(f"epoch [{epoch+1}/{epochs}], Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss.item():.4f}")

epoch [10/100], Train Loss: 12.8780, Val Loss: 8.0260
epoch [20/100], Train Loss: 9.9615, Val Loss: 8.1898
epoch [30/100], Train Loss: 8.1766, Val Loss: 7.8735
epoch [40/100], Train Loss: 7.6249, Val Loss: 9.2107
epoch [50/100], Train Loss: 7.6125, Val Loss: 9.5849
epoch [60/100], Train Loss: 6.3226, Val Loss: 7.6740
epoch [70/100], Train Loss: 6.4494, Val Loss: 8.4517
epoch [80/100], Train Loss: 5.7400, Val Loss: 7.6815
epoch [90/100], Train Loss: 5.5336, Val Loss: 8.8921
epoch [100/100], Train Loss: 4.6624, Val Loss: 9.0712


In [13]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    test_loss = criterion(test_pred, y_test)
    print(f"Test MSE Loss: {test_loss.item():.4f}") # MSE 출력
    print(f"실제값(예시): {y_test[0].item():.2f}, 예측값: {test_pred[0].item():.2f}")

mae = torch.mean(torch.abs(test_pred - y_test)) # MAE 계산
print(f"Test MAE (오차 평균): {mae.item():.4f}")

Test MSE Loss: 9.2565
실제값(예시): 23.60, 예측값: 27.78
Test MAE (오차 평균): 2.2356


### 클래스 버전

In [14]:
class BostonHousingModel(nn.Module):
    def __init__(self, input_dim):
        super(BostonHousingModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self, x):
        return self.model(x)

In [15]:
model = BostonHousingModel(input_dim=13)

In [16]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [17]:
epochs = 100
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = criterion(val_pred, y_val)

    if (epoch + 1) % 10 == 0:
        print(f"epoch [{epoch+1}/{epochs}], Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss.item():.4f}")

epoch [10/100], Train Loss: 12.8731, Val Loss: 9.3361
epoch [20/100], Train Loss: 10.3650, Val Loss: 7.2459
epoch [30/100], Train Loss: 9.2381, Val Loss: 7.7945
epoch [40/100], Train Loss: 7.5111, Val Loss: 7.9729
epoch [50/100], Train Loss: 6.6310, Val Loss: 7.3444
epoch [60/100], Train Loss: 6.5914, Val Loss: 8.9646
epoch [70/100], Train Loss: 5.9201, Val Loss: 7.9135
epoch [80/100], Train Loss: 5.0528, Val Loss: 8.1044
epoch [90/100], Train Loss: 4.8288, Val Loss: 7.7579
epoch [100/100], Train Loss: 5.3219, Val Loss: 8.2849


In [18]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    test_loss = criterion(test_pred, y_test)
    print(f"Test MSE Loss: {test_loss.item():.4f}") # MSE 출력
    print(f"실제값(예시): {y_test[0].item():.2f}, 예측값: {test_pred[0].item():.2f}")

mae = torch.mean(torch.abs(test_pred - y_test)) # MAE 계산
print(f"Test MAE (오차 평균): {mae.item():.4f}")

Test MSE Loss: 7.0924
실제값(예시): 23.60, 예측값: 28.00
Test MAE (오차 평균): 2.1082
